# 07 - Improved Context Formatting and Source-Aware Answers

## Purpose

This notebook improves the PDF Q&A chatbot by formatting retrieved context more clearly and returning source information with each answer.

The previous notebook created a working GPT-based Q&A chain. This notebook adds a cleaner context structure and a helper function that returns both the answer and the retrieved source metadata.

## Main Changes in Version 2

This version adds:

- Cleaner retrieved context formatting
- Source metadata display
- Section and page information in the final answer
- A more transparent Q&A response
- Better debugging visibility for retrieved chunks

## Main Outputs

- `format_context_with_sources`
- `ask_pdf_v2_with_sources`

## Notebook Flow

```text
User question
        ↓
Retriever returns documents
        ↓
Format context with metadata
        ↓
Send context and question to GPT
        ↓
Return answer
        ↓
Display sources used

In [0]:
%pip install langchain langchain-community langchain-openai pypdf tiktoken chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
import os

os.environ["OPENAI_API_KEY"] = "openai_api_key"

In [0]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, TokenTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
import shutil
import os
import time

pdf_path = "/Volumes/workspace/365pdf/365pdfupload/Introduction_to_Tableau.pdf"

loader_pdf = PyPDFLoader(pdf_path)
docs_list = loader_pdf.load()

markdown_transcript = ""

for doc in docs_list:
    page_number = doc.metadata.get("page", "unknown")
    page_text = doc.page_content

    markdown_transcript += f"\n\n# Introduction to Tableau\n"
    markdown_transcript += f"## Page {page_number}\n\n"
    markdown_transcript += page_text

headers_to_split_on = [
    ("#", "Section Title"),
    ("##", "Page Title")
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

docs_list_md_split = md_splitter.split_text(markdown_transcript)

print("Header-split documents:", len(docs_list_md_split))

chat_formatter = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

formatting_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a document-cleaning assistant.

Your task is to clean text extracted from a PDF.

Rules:
- Fix broken line breaks and awkward spacing.
- Improve punctuation and capitalization where needed.
- Keep the original meaning unchanged.
- Do not add new information.
- Do not summarize.
- Do not remove important details.
- Return only the cleaned text.
"""
    ),
    (
        "human",
        """
Clean and format the following extracted PDF text:

{text}
"""
    )
])

formatting_chain = formatting_prompt | chat_formatter | StrOutputParser()

formatted_docs_list = []

for i, doc in enumerate(docs_list_md_split):
    print(f"Formatting document {i + 1} of {len(docs_list_md_split)}")

    text_to_format = doc.page_content[:3000]

    formatted_text = formatting_chain.invoke({
        "text": text_to_format
    })

    formatted_doc = Document(
        page_content=formatted_text,
        metadata=doc.metadata
    )

    formatted_docs_list.append(formatted_doc)

    time.sleep(0.5)

print("Formatted documents:", len(formatted_docs_list))

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50
)

docs_list_tokens_split = token_splitter.split_documents(formatted_docs_list)

print("Token-split documents:", len(docs_list_tokens_split))

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
from langchain_community.vectorstores import Chroma
import uuid
import os

persist_directory = f"/tmp/chroma_intro_tableau_v2_{uuid.uuid4().hex}"

os.makedirs(persist_directory, exist_ok=True)

print("Using fresh Chroma directory:")
print(persist_directory)

vectorstore = Chroma.from_documents(
    documents=docs_list_tokens_split,
    embedding=embedding,
    persist_directory=persist_directory
)

print("Chroma vector store rebuilt successfully")

try:
    print("Chroma collection count:", vectorstore._collection.count())
except Exception as e:
    print("Could not count collection:", e)

In [0]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 8,
        "lambda_mult": 0.7
    }
)

print("MMR retriever created successfully")

In [0]:
test_question = "What is Tableau used for?"
test_docs = retriever.invoke(test_question)

print("Retrieved docs:", len(test_docs))

for i, doc in enumerate(test_docs, start=1):
    print(f"\nDocument {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:500])
    print("-" * 80)

In [0]:
def format_context_with_sources(docs):
    context_text = ""
    source_lines = []

    for i, doc in enumerate(docs, start=1):
        section_title = doc.metadata.get("Section Title", "Unknown section")
        page_title = doc.metadata.get("Page Title", "Unknown page")

        context_text += f"""
Document {i}
Section Title: {section_title}
Page Title: {page_title}

Content:
{doc.page_content}

--------------------
"""

        source_lines.append(
            f"- Document {i}: Section Title='{section_title}', Page Title='{page_title}'"
        )

    return context_text, source_lines

In [0]:
context_text, source_lines = format_context_with_sources(test_docs)

print("Formatted context preview:")
print(context_text[:2000])

print("\nSources:")
for source in source_lines:
    print(source)

In [0]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

rag_prompt_sources = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful PDF question-answering assistant.

Answer the user's question using only the retrieved PDF context.

Rules:
- Use only the provided context.
- Do not use outside knowledge.
- If the context does not contain the answer, say:
  "I could not find this information in the uploaded PDF."
- Keep the answer clear and beginner-friendly.
- Do not mention internal implementation details.
- Do not invent page numbers, sections, or facts.
"""
    ),
    (
        "human",
        """
Retrieved PDF context:

{context}

User question:

{question}
"""
    )
])

print("Source-aware prompt created")

In [0]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

def ask_pdf_v2_with_sources(question):
    docs = retriever.invoke(question)

    if len(docs) == 0:
        return "I could not find relevant information in the uploaded PDF."

    context_text, source_lines = format_context_with_sources(docs)

    messages = rag_prompt_sources.invoke({
        "context": context_text,
        "question": question
    })

    answer = chat.invoke(messages)
    answer_text = output_parser.invoke(answer)

    final_response = f"""
Answer:
{answer_text}

Sources used:
{chr(10).join(source_lines)}
"""

    return final_response

In [0]:
print(ask_pdf_v2_with_sources("What is Tableau used for?"))

In [0]:
print(ask_pdf_v2_with_sources("What are dimensions and measures in Tableau?"))

In [0]:
print(ask_pdf_v2_with_sources("How does Tableau help with data visualization?"))

In [0]:
print(ask_pdf_v2_with_sources("What is the capital of Japan?"))